# 1️⃣ Importing Library

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline
import joblib

# 2️⃣ Loading Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/olist_data'
df_merged = pd.read_csv(f'{BASE_PATH}/data.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3️⃣ Machine Learning

In [ ]:
features = ['price', 'freight_value']
cat_cols = ['product_category_name_english', 'customer_state', 'seller_state']

x = df_merged[features + cat_cols].copy()
y = df_merged['is_late']

# 결측치 처리
x[cat_cols] = x[cat_cols].fillna('Unknown')
x[features] = x[features].fillna(0)

try:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse=False)

preproc = ColumnTransformer([
    ('num', 'passthrough', features),
    ('cat', cat_transform, cat_cols)
])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=24,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([
    ('preprocessor', preproc),
    ('model', clf)
])

pipe.fit(x_train, y_train)

joblib.dump(pipe, "delay_prediction_model.pkl")
print("-" * 120)

pred = pipe.predict(x_test)
pred_proba = pipe.predict_proba(x_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, pred))
print(f"ROC-AUC : {roc_auc_score(y_test, pred_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, pred_proba):.4f}")
print(f"Train Score : {pipe.score(x_train, y_train):.4f}")
print(f"Test Score  : {pipe.score(x_test, y_test):.4f}")
print("_" * 120)


📂📂📂 Model is saved.
------------------------------------------------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.83      0.88     22613
           1       0.15      0.36      0.21      1911

    accuracy                           0.79     24524
   macro avg       0.55      0.59      0.55     24524
weighted avg       0.88      0.79      0.83     24524

ROC-AUC : 0.6552
PR-AUC  : 0.1463
Train Score : 0.8401
Test Score  : 0.7949
________________________________________________________________________________________________________________________
